# __Demo: Role-Based Prompting for Engineering Tasks with Hugging Face__

## __Steps to Perform:__
Step 1: Set up the Environment and Initialize `Qwen/Qwen2.5-0.5B-Instruct`

Step 2: Define a Function to Get Completion

Step 3: Assign the Code Reviewer Role

Step 4: Assign the QA Engineer Role

Step 5: Assign the Technical Writer Role

Step 6: Compare Against the Same Prompt With No Role

### __Step 1: Set up the Environment and Initialize `Qwen/Qwen2.5-0.5B-Instruct`__
- Installs required libraries (`transformers`, `torch`, `accelerate`).
- Loads `Qwen/Qwen2.5-0.5B-Instruct` onto the GPU in 16-bit half precision (`torch.float16`).
- No API key, token, or billing configuration is required.

In [1]:
!pip install -q transformers torch accelerate

import os
import torch
from transformers import pipeline

# Initialize text-generation pipeline on GPU with float16
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
print("✅ Qwen/Qwen2.5-0.5B-Instruct loaded successfully on GPU.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ Qwen/Qwen2.5-0.5B-Instruct loaded successfully on GPU.


### __Step 2: Define a Function to Get Completion__
- Constructs a message formatted as a chat role dictionary with the user's prompt.
- Calls the Hugging Face pipeline with greedy decoding (`do_sample=False`) for deterministic (non-random) engineering responses.

In [2]:
def get_completion(prompt, max_tokens=250):
    messages = [{"role": "user", "content": prompt}]
    output = generator(messages, max_new_tokens=max_tokens, do_sample=False)
    return output[0]["generated_text"][-1]["content"].strip()

print("✅ Completion helper ready.")

✅ Completion helper ready.


### __Step 3: Assign the Code Reviewer Role__
Assigning the model a specific role focuses its reasoning and vocabulary on exactly the perspective the task needs -- here, a security- and correctness-focused code reviewer looking at a real diff.

In [3]:
code_diff = """
- def get_user(user_id):
-     query = "SELECT * FROM users WHERE id = " + user_id
-     return db.execute(query)
+ def get_user(user_id):
+     query = "SELECT * FROM users WHERE id = ?"
+     return db.execute(query, (user_id,))
"""

reviewer_prompt = f"""
You are a senior Python code reviewer. Review this pull request diff for correctness, security, and style issues before it merges. List each issue found, or state that none were found.

Diff:
{code_diff}
"""
print("=== Step 3: Code Reviewer Role Output ===")
print(get_completion(reviewer_prompt))


=== Step 3: Code Reviewer Role Output ===
None of the issues in the provided pull request diff have been identified as they do not affect the core functionality or structure of the code. The two functions `get_user` are identical, with only minor differences in their parameter names and docstrings. This means there is no need to review any specific security or style issues related to these changes.


### __Step 4: Assign the QA Engineer Role__
The same underlying code can be handed to a different role to get a different, equally useful perspective -- here, a QA engineer focused on test coverage rather than code correctness.

In [4]:
function_under_test = """
def get_user(user_id):
    query = "SELECT * FROM users WHERE id = ?"
    return db.execute(query, (user_id,))
"""

qa_prompt = f"""
You are a QA engineer designing edge-case tests. List every boundary condition this function should be tested against, as a numbered list. Do not write the test code, only the conditions to cover.

Function:
{function_under_test}
"""
print("=== Step 4: QA Engineer Role Output ===")
print(get_completion(qa_prompt))


=== Step 4: QA Engineer Role Output ===
1. user_id is 0
2. user_id is None
3. user_id is an empty string
4. user_id is a negative number
5. user_id is greater than the number of rows in the database
6. user_id is less than or equal to 0
7. user_id is within a valid range defined by the database schema
8. user_id is outside the valid range defined by the database schema
9. user_id is a non-numeric value
10. user_id is a non-string value
11. user_id is a non-zero value
12. user_id is a non-negative integer
13. user_id is a non-positive integer
14. user_id is a non-float value
15. user_id is a non-string representation of a date/time
16. user_id is a non-integer value
17. user_id is a non-binary value
18. user_id is a non-boolean value
19. user_id is a non-list value
20. user_id is a non-tuple value
21. user_id is a non-set value
22. user_id is a non


### __Step 5: Assign the Technical Writer Role__
A third role -- technical writer -- turns the same function into documentation a new engineer could understand quickly.

In [5]:
writer_prompt = f"""
You are a technical writer. Turn this function's behavior into a clear docstring a new engineer could
understand in one read.

Function:
{function_under_test}
"""
print("=== Step 5: Technical Writer Role Output ===")
print(get_completion(writer_prompt))


=== Step 5: Technical Writer Role Output ===
```python
# This function retrieves a user from the database based on their ID.
# It takes an integer parameter 'user_id' and returns a dictionary containing all columns of the 'users' table where the 'id' matches the provided user ID.
```

This docstring provides a clear explanation of what the function does, its parameters, and its return value. The docstring is concise yet informative, making it easy for engineers to understand how to use the function effectively.


### __Step 6: Compare Against the Same Prompt With No Role__
To see the effect of the role itself, ask essentially the same underlying question with no persona assigned, and compare the tone and focus of the response against Step 3.

In [6]:
no_role_prompt = f"""
Look at this code diff and say what you think.

Diff:
{code_diff}
"""
print("=== Step 6: Baseline Output (No Role Assigned) ===")
print(get_completion(no_role_prompt))


=== Step 6: Baseline Output (No Role Assigned) ===
The provided code differences involve minor changes to the SQL query string used in the `get_user` function. Specifically, the first version of the function uses an explicit placeholder (`?`) for the user ID parameter, while the second version uses a tuple unpacking syntax (`(user_id,)`). This difference does not significantly alter the functionality or behavior of the function but may affect how it interacts with the database.

In general, both versions should work as expected, assuming that the `db` object is properly initialized and configured to execute the queries. The choice between using a placeholder versus a tuple depends on personal preference or specific requirements, such as whether you prefer readability or maintainability.


## __Conclusion__
The role assigned to the model doesn't change what it knows -- it changes what it pays attention to and how it phrases the response. The same diff produced a security- and correctness-focused review under the code-reviewer role, a coverage-focused list under the QA-engineer role, and clear documentation under the technical-writer role, each more useful for its purpose than the unfocused response with no role assigned.